# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subikshasrig/FlyrankMLInternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The first signal check tested staleness using days since the last update. The 180+ day group had higher observed CTR than the newer group, so the signal was judged OPPOSITE and was not used as a positive input to the baseline.
The second check compared CTR below 1% with CTR at or above 1%. The lower-CTR group showed a weaker observed average position, so this signal was judged CONFIRMED.

The baseline therefore prioritizes pages with low observed CTR and weak average position. Higher observed 90-day impressions increase the priority of pages meeting both conditions.

Reason code:
LOW_CTR_WEAK_POSITION - low observed CTR combined with weak average position.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

required_cols = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

missing = [
    col for col in required_cols
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

staleness_check = df.copy()

staleness_check["staleness_bucket"] = (
    staleness_check["days_since_last_update"]
    >= 180
)

staleness_table = (
    staleness_check
    .groupby("staleness_bucket")
    .agg(
        n=("ctr", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        mean_impressions_90d=("impressions_90d", "mean")
    )
    .reset_index()
)

staleness_table["staleness_bucket"] = (
    staleness_table["staleness_bucket"]
    .map({
        True: "180+ days since update",
        False: "<180 days since update"
    })
)

print("=" * 70)
print("SIGNAL CHECK 1 — STALENESS")
print("=" * 70)

display(staleness_table)

print("Verdict: OPPOSITE")

ctr_position_check = df.copy()

ctr_position_check["ctr_bucket"] = (
    ctr_position_check["ctr"] >= 1.0
)

ctr_position_table = (
    ctr_position_check
    .groupby("ctr_bucket")
    .agg(
        n=("ctr", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median"),
        mean_position=("avg_position", "mean"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

ctr_position_table["ctr_bucket"] = (
    ctr_position_table["ctr_bucket"]
    .map({
        False: "<1% CTR",
        True: ">=1% CTR"
    })
)

print()
print("=" * 70)
print("SIGNAL CHECK 2 — CTR VS POSITION")
print("=" * 70)

display(ctr_position_table)

print("Verdict: CONFIRMED")

SIGNAL CHECK 1 — STALENESS


,staleness_bucket,n,mean_ctr,median_ctr,mean_impressions_90d
0,<180 days since update,29826,0.492167,0.07,5223.864514
1,180+ days since update,174,3.693276,0.00,1172.448276


Verdict: OPPOSITE

SIGNAL CHECK 2 — CTR VS POSITION


,ctr_bucket,n,mean_ctr,median_ctr,mean_position,median_position
0,<1% CTR,28291,0.151639,0.05,16.698798,11.1
1,>=1% CTR,1709,6.455225,1.79,10.442188,7.2


Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
# Show variables currently available in the notebook
%whos

Variable               Type         Data/Info
---------------------------------------------
DATA_PATH              PosixPath    data/raw/content_refresh_anonymized.csv
Path                   type         <class 'pathlib.Path'>
confidence_note        function     <function confidence_note at 0x7b75e9dc2660>
ctr_position_check     DataFrame                     content_<...>[30000 rows x 45 columns]
ctr_position_table     DataFrame      ctr_bucket      n  mean<...>0.442188              7.2
df                     DataFrame                     content_<...>[30000 rows x 44 columns]
future_columns         list         n=0
leakage_free           bool         True
missing                list         n=0
np                     module       <module 'numpy' from '/us<...>kages/numpy/__init__.py'>
output                 DataFrame            rank            c<...>n[30000 rows x 9 columns]
output_cols            list         n=9
output_path            PosixPath    work/outputs/baseline_action_score.

In [18]:
from pathlib import Path
for path in Path(".").rglob("*"):
    if path.is_file():
        print(path)

LICENSE
AGENTS.md
CLAUDE.md
SETUP.md
.gitignore
requirements.txt
README.md
DATA_USE.md
GUIDE.md
outputs/refresh_queue_sample.csv
outputs/model_report.md
skills/README.md
work/capstone_report_template.md
work/README.md
.git/HEAD
.git/description
.git/config
.git/index
.git/packed-refs
docs/ml-core-foundation-framework.md
docs/ml-intern-dataset-and-lane-guide.md
docs/flyrank-seo-research-march-2026.pdf
docs/intern-free-tooling-guide.md
docs/data-dictionary.md
notebooks/01_first_look_and_discovery.ipynb
notebooks/03_working_with_the_full_release.ipynb
notebooks/02_your_first_readable_model.ipynb
submission/README.md
submission/paper_url.txt
scripts/03_train_model.py
scripts/01_prepare_features.py
scripts/02_baseline_score.py
scripts/ml_utils.py
scripts/04_evaluate_and_export.py
scripts/run_all.py
scripts/05_build_pdf_report.py
data/raw/content_refresh_anonymized.csv
outputs/charts/confidence_mix.svg
outputs/charts/action_mix.svg
outputs/charts/trend_distribution.svg
outputs/charts/top_fea

In [19]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.44 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [20]:
from pathlib import Path

print("Current directory:")
print(Path.cwd())

print("\nML-07 notebook exists:")
print(Path("work/notebooks/w04_baseline_score.ipynb").exists())

print("\nCSV files:")
for p in Path(".").rglob("*.csv"):
    print(" ", p)

print("\nRelevant skill files:")
for p in Path("skills").rglob("*"):
    if p.is_file():
        print(" ", p)

Current directory:
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter

ML-07 notebook exists:
True

CSV files:
  outputs/refresh_queue_sample.csv
  data/raw/content_refresh_anonymized.csv

Relevant skill files:
  skills/README.md
  skills/writing-honest-claims/SKILL.md
  skills/directing-your-ai-assistant/SKILL.md
  skills/hunting-leakage-and-validating/SKILL.md
  skills/writing-research-papers/SKILL.md
  skills/auditing-signals/SKILL.md
  skills/querying-big-datasets/SKILL.md
  skills/training-honest-models/SKILL.md
  skills/building-baselines/SKILL.md
  skills/deploying-static-pages/SKILL.md
  skills/writing-data-contracts/SKILL.md
  skills/framing-ml-problems/SKILL.md
  skills/flyrank/flyrank-context/SKILL.md
  skills/flyrank/flyrank-data/SKILL.md


In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

required_cols = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["low_ctr"] = (
    df["ctr"] < 1.0
).astype(int)

df["weak_position"] = (
    df["avg_position"] > 10
).astype(int)

df["ctr_severity"] = (
    1 - np.minimum(df["ctr"] / 1.0, 1)
)

df["position_severity"] = (
    np.minimum(df["avg_position"] / 20.0, 2)
)

df["score"] = np.where(
    (df["low_ctr"] == 1) & (df["weak_position"] == 1),
    df["impressions_90d"].fillna(0)
    * df["ctr_severity"]
    * df["position_severity"],
    0
)

df["action"] = np.where(
    df["score"] > 0,
    "REFRESH_REVIEW",
    "NO_ACTION"
)

df["reason_code"] = np.where(
    df["score"] > 0,
    "LOW_CTR_WEAK_POSITION",
    "NONE"
)

df = df.sort_values(
    by=[
        "score",
        "impressions_90d"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

output_cols = [
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

output = df[output_cols].copy()

output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

output.to_csv(
    output_path,
    index=False
)

print("Rows ranked:", len(output))
print(
    "Rows flagged for refresh review:",
    (output["action"] == "REFRESH_REVIEW").sum()
)
print("CSV written to:", output_path)
print()
print("Action counts:")
print(output["action"].value_counts())
print()
print("Top 20:")
display(output.head(20))

Rows ranked: 30000
Rows flagged for refresh review: 15265
CSV written to: work/outputs/baseline_action_score.csv

Action counts:
action
REFRESH_REVIEW    15265
NO_ACTION         14735
Name: count, dtype: int64

Top 20:


,rank,content_id,score,action,reason_code,impressions_90d,ctr,avg_position,days_since_last_update
0,1,content_2cb567c3c89b,497229.27300,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,497727,0.10,22.2,48
1,2,content_2dba2b1f9536,488686.43970,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,443434,0.21,27.9,104
2,3,content_a023517539fe,423813.06000,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,214047,0.01,85.8,20
3,4,content_b28d1efd668f,352929.09120,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,286608,0.06,26.2,104
4,5,content_ff94c9b6b411,300610.00320,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,228566,0.04,27.4,20
5,6,content_813e88069237,287607.01540,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,233561,0.06,26.2,104
6,7,content_66b4046cc144,280487.09150,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,217415,0.03,26.6,20
7,8,content_54baba704595,258621.66000,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,130617,0.01,47.0,104
8,9,content_88d367c507a3,251389.44000,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,130932,0.04,40.1,104
9,10,content_b511d4bc4ad2,247036.22550,REFRESH_REVIEW,LOW_CTR_WEAK_POSITION,205915,0.14,27.9,104


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

required_cols = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["ctr_bucket"] = np.where(
    df["ctr"] < 1.0,
    "<1% CTR",
    ">=1% CTR"
)

df["low_ctr"] = (
    df["ctr"] < 1.0
).astype(int)

df["score"] = (
    df["low_ctr"]
    * df["impressions_90d"].fillna(0)
)

df["action"] = np.where(
    df["score"] > 0,
    "REFRESH_REVIEW",
    "NO_ACTION"
)

df["reason_code"] = np.where(
    df["score"] > 0,
    "LOW_CTR",
    "NONE"
)

df = df.sort_values(
    by=[
        "score",
        "impressions_90d"
    ],
    ascending=[
        False,
        False
    ]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

output_cols = [
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

output = df[output_cols].copy()

output_path = Path(
    "work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

output.to_csv(
    output_path,
    index=False
)

print("Rows ranked:", len(output))
print(
    "Rows flagged for refresh review:",
    (output["action"] == "REFRESH_REVIEW").sum()
)
print("CSV written to:", output_path)
print()
print("Action counts:")
print(output["action"].value_counts())
print()
print("Top 20:")
display(output.head(20))

Rows ranked: 30000
Rows flagged for refresh review: 28291
CSV written to: work/outputs/baseline_action_score.csv

Action counts:
action
REFRESH_REVIEW    28291
NO_ACTION          1709
Name: count, dtype: int64

Top 20:


,rank,content_id,score,action,reason_code,impressions_90d,ctr,avg_position,days_since_last_update
0,1,content_5fe46e04994d,517715,REFRESH_REVIEW,LOW_CTR,517715,0.14,4.2,104
1,2,content_aaef01a50def,517109,REFRESH_REVIEW,LOW_CTR,517109,0.25,5.4,22
2,3,content_8c19996aa890,509252,REFRESH_REVIEW,LOW_CTR,509252,0.15,2.5,20
3,4,content_2cb567c3c89b,497727,REFRESH_REVIEW,LOW_CTR,497727,0.10,22.2,48
4,5,content_4c36c775b818,463103,REFRESH_REVIEW,LOW_CTR,463103,0.41,2.3,20
5,6,content_2dba2b1f9536,443434,REFRESH_REVIEW,LOW_CTR,443434,0.21,27.9,104
6,7,content_1a9e894be2e2,416180,REFRESH_REVIEW,LOW_CTR,416180,0.23,4.0,22
7,8,content_2c2606c5d176,347399,REFRESH_REVIEW,LOW_CTR,347399,0.53,4.2,104
8,9,content_db5989a78dd3,345111,REFRESH_REVIEW,LOW_CTR,345111,0.21,5.4,20
9,10,content_44e481c8f55b,312694,REFRESH_REVIEW,LOW_CTR,312694,0.65,1.4,20


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 are all REFRESH_REVIEW recommendations with the reason code LOW_CTR_WEAK_POSITION. The ranking gives more weight to pages with higher observed impressions, lower CTR, and weaker average position.
Each recommendation is decision-support rather than a confirmed content problem. A recommendation could be wrong if query intent, SERP context, competition, or other factors explain the observed CTR and position.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = output.head(20).copy()

def confidence_note(row):
    if row["ctr"] <= 0.05 and row["avg_position"] >= 30:
        return "High: very low CTR and weak position."
    elif row["ctr"] <= 0.20 and row["avg_position"] >= 20:
        return "High: low CTR and weak position."
    elif row["ctr"] <= 0.50 and row["avg_position"] >= 15:
        return "Moderate: both signals indicate review."
    else:
        return "Moderate: meets both baseline conditions."

def wrong_reason(row):
    if row["avg_position"] >= 50:
        return "Could be wrong if the page targets highly competitive queries."
    elif row["ctr"] <= 0.05:
        return "Could be wrong if SERP context or query intent explains the very low CTR."
    elif row["avg_position"] <= 15:
        return "Could be wrong if the page's position and CTR are appropriate for its query intent."
    else:
        return "Could be wrong if query mix or SERP context explains the observed performance."

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

review_cols = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_cols])

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,REFRESH_REVIEW,LOW_CTR,Moderate: meets both baseline conditions.,Could be wrong if the page's position and CTR ...
1,2,content_aaef01a50def,REFRESH_REVIEW,LOW_CTR,Moderate: meets both baseline conditions.,Could be wrong if the page's position and CTR ...
2,3,content_8c19996aa890,REFRESH_REVIEW,LOW_CTR,Moderate: meets both baseline conditions.,Could be wrong if the page's position and CTR ...
3,4,content_2cb567c3c89b,REFRESH_REVIEW,LOW_CTR,High: low CTR and weak position.,Could be wrong if query mix or SERP context ex...
4,5,content_4c36c775b818,REFRESH_REVIEW,LOW_CTR,Moderate: meets both baseline conditions.,Could be wrong if the page's position and CTR ...
5,6,content_2dba2b1f9536,REFRESH_REVIEW,LOW_CTR,Moderate: both signals indicate review.,Could be wrong if query mix or SERP context ex...
6,7,content_1a9e894be2e2,REFRESH_REVIEW,LOW_CTR,Moderate: meets both baseline conditions.,Could be wrong if the page's position and CTR ...
7,8,content_2c2606c5d176,REFRESH_REVIEW,LOW_CTR,Moderate: meets both baseline conditions.,Could be wrong if the page's position and CTR ...
8,9,content_db5989a78dd3,REFRESH_REVIEW,LOW_CTR,Moderate: meets both baseline conditions.,Could be wrong if the page's position and CTR ...
9,10,content_44e481c8f55b,REFRESH_REVIEW,LOW_CTR,Moderate: meets both baseline conditions.,Could be wrong if the page's position and CTR ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weaker picks are pages where the recommendation may be less compelling despite meeting the baseline rule. In particular, pages with CTR closer to 1% or average position closer to 10 have less severe signals than the strongest picks.

The baseline uses only observed 90-day impressions, CTR, average position, and days since last update. It does not use product flags, future-window measurements, or label-derived inputs.
The baseline should therefore be treated as decision-support rather than proof that a page needs a refresh.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
weak_picks = top20[
    (top20["ctr"] > 0.50) |
    (top20["avg_position"] < 15)
].copy()

print("Weak picks:")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "ctr",
            "avg_position",
            "impressions_90d",
            "action",
            "reason_code"
        ]
    ]
)

print()
print("LEAKAGE CHECK")

future_columns = [
    col for col in df.columns
    if any(
        term in col.lower()
        for term in [
            "future",
            "next_",
            "label",
            "target",
            "outcome"
        ]
    )
]

product_flag_columns = [
    col for col in df.columns
    if "flag" in col.lower()
    or "product" in col.lower()
]

print("Future/label-like columns found:")
print(future_columns)

print()
print("Product/flag-like columns found:")
print(product_flag_columns)

used_columns = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

print()
print("Columns used by baseline:")
print(used_columns)

leakage_free = (
    len(future_columns) == 0
    and len(product_flag_columns) == 0
)

print()
print("Leakage check passed:", leakage_free)

Weak picks:


,rank,content_id,ctr,avg_position,impressions_90d,action,reason_code
0,1,content_5fe46e04994d,0.14,4.2,517715,REFRESH_REVIEW,LOW_CTR
1,2,content_aaef01a50def,0.25,5.4,517109,REFRESH_REVIEW,LOW_CTR
2,3,content_8c19996aa890,0.15,2.5,509252,REFRESH_REVIEW,LOW_CTR
4,5,content_4c36c775b818,0.41,2.3,463103,REFRESH_REVIEW,LOW_CTR
6,7,content_1a9e894be2e2,0.23,4.0,416180,REFRESH_REVIEW,LOW_CTR
7,8,content_2c2606c5d176,0.53,4.2,347399,REFRESH_REVIEW,LOW_CTR
8,9,content_db5989a78dd3,0.21,5.4,345111,REFRESH_REVIEW,LOW_CTR
9,10,content_44e481c8f55b,0.65,1.4,312694,REFRESH_REVIEW,LOW_CTR
10,11,content_cb112fce36be,0.16,5.6,309910,REFRESH_REVIEW,LOW_CTR
11,12,content_9532f197bbc8,0.87,2.0,309192,REFRESH_REVIEW,LOW_CTR



LEAKAGE CHECK
Future/label-like columns found:
[]

Product/flag-like columns found:
[]

Columns used by baseline:
['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update']

Leakage check passed: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.